#Reading From Bronze Table

# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType, DoubleType, IntegerType
from pyspark.sql.functions import trim, col
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window


## Product Information Table

In [0]:
df = spark.table("dev_project.bronze.crm_prd_info")

In [0]:
df.display()

### Data Transformation

In [0]:

string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
print("String columns:", string_cols)

for col_name in string_cols:
    df = df.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)) == "", None)  # blank string → NULL
         .otherwise(F.trim(F.col(col_name)))
    )
    
display(df.select("prd_key", "prd_line", "prd_cost", "prd_start_dt", "prd_end_dt").limit(10))

### Produck Key Parsing

In [0]:
parts = F.split(col("prd_key"), "-")

df = df.withColumn("cat_id", F.concat_ws("_", parts[0], parts[1])) \
       .withColumn("product_code", F.get(parts, 3)) \
       .withColumn("size", F.get(parts, 4))
# Derive shortened product key to match sales transaction format
# 5-segment keys: take segments [2][3][4]
# 4-segment keys: take segments [2][3]
parts = F.split(F.col("prd_key"), "-")

df = df.withColumn(
    "prd_key_short",
    F.when(
        F.size(parts) == 5,
        F.concat_ws("-",
            F.get(parts, 2),
            F.get(parts, 3),
            F.get(parts, 4)
        )
    ).when(
        F.size(parts) == 4,
        F.concat_ws("-",
            F.get(parts, 2),
            F.get(parts, 3)
        )
    )
)

# Fix source system inconsistency — CRM uses CO_PE for pedals, ERP uses CO_PD
df = df.withColumn(
    "cat_id",
    F.when(F.col("cat_id") == "CO_PE", "CO_PD")
     .otherwise(F.col("cat_id"))
)

### Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

### Product Line Normalization

In [0]:

df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

### Date Handling

In [0]:
# Clean dirty end date values — trim first, then check
# isNull() used explicitly — isin() silently ignores nulls in Spark
df = df.withColumn(
    "prd_end_dt",
    F.when(
        F.col("prd_end_dt").isNull() |
        (F.trim(F.col("prd_end_dt")) == "") |
        (F.trim(F.col("prd_end_dt")) == "-"),
        None
    ).otherwise(F.trim(F.col("prd_end_dt")))
)

# Cast both date columns now that end date is clean
df = df.withColumn("prd_start_dt", F.to_date("prd_start_dt")) \
       .withColumn("prd_end_dt",   F.to_date("prd_end_dt"))

# Derive is_current from cast column — null end date = current record
# This MUST happen before nulling bad dates below
df = df.withColumn(
    "is_current",
    F.when(F.col("prd_end_dt").isNull(), True).otherwise(False)
)

# Null out end_date < start_date — source system bug
# is_current already captured above so this cannot corrupt it
df = df.withColumn(
    "prd_end_dt",
    F.when(F.col("prd_end_dt") < F.col("prd_start_dt"), None)
     .otherwise(F.col("prd_end_dt"))
)

### Dedup / Latest Record Logic

In [0]:
window_dedup = Window.partitionBy("prd_id", "prd_start_dt") \
                     .orderBy(F.col("_file_name").desc())

df = df.withColumn("_row_num", F.row_number().over(window_dedup)) \
       .filter(F.col("_row_num") == 1) \
       .drop("_row_num")

### SCD2 Version:


In [0]:
window_version = Window.partitionBy("prd_id").orderBy(F.col("prd_start_dt").asc())

df = df.withColumn("product_version", F.row_number().over(window_version))

### Surrogate Key:


In [0]:
df = df.withColumn(
    "product_surrogate_key",
    F.sha2(
        F.concat_ws("||",
            F.col("prd_id").cast("string"),
            F.col("prd_start_dt").cast("string")
        ),
        256
    )
)

### Rename + Final Select:

In [0]:
RENAME_MAP_PRD = {
    "prd_id":       "product_id",
    "prd_key":      "product_key",
    "prd_nm":       "product_name",
    "prd_cost":     "product_cost",
    "prd_line":     "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt":   "end_date"
}

for old_col, new_col in RENAME_MAP_PRD.items():
    df = df.withColumnRenamed(old_col, new_col)

df_products_silver = df.select(
    "product_surrogate_key",
    "product_id",
    "product_key",
    "prd_key_short",        
    "product_name",
    "cat_id",
    "product_code",
    "size",
    "product_cost",
    "product_line",
    "start_date",
    "end_date",
    "is_current",
    "product_version"
)

### Validation

In [0]:
print("\n" + "="*60)
print(" DATA QUALITY VALIDATION")
print("="*60)
print(f"Total records:        {df_products_silver.count():,}")
print(f"Current versions:     {df_products_silver.filter(F.col('is_current') == True).count():,}")
print(f"Historical versions:  {df_products_silver.filter(F.col('is_current') == False).count():,}")

dup_check = df_products_silver.groupBy("product_id", "start_date").count().filter("count > 1")
print(f"Duplicate keys:       {dup_check.count()}")
if dup_check.count() > 0:
    dup_check.show()

invalid_dates = df_products_silver.filter(
    F.col("end_date").isNotNull() & (F.col("end_date") < F.col("start_date"))
).count()
print(f"Invalid dates:        {invalid_dates}")

null_ids = df_products_silver.filter(F.col("product_id").isNull()).count()
print(f"Null product IDs:     {null_ids}")

null_costs = df_products_silver.filter(F.col("product_cost").isNull()).count()
print(f"Null costs:           {null_costs}")
print("="*60 + "\n")

### Write into Product Table

In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = "dev_project.silver.crm_prd_info"
today = F.current_date()

# --- FIRST RUN: Table doesn't exist yet — write and exit ---
if not spark.catalog.tableExists(TARGET_TABLE):
    df_products_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(TARGET_TABLE)
    print("Initial load complete — table created.")

else:
    silver = DeltaTable.forName(spark, TARGET_TABLE)

    # Step 1: Hash business columns on incoming batch
    df_incoming = df_products_silver.withColumn(
        "incoming_hash",
        F.sha2(F.concat_ws("||",
            F.col("product_name"),
            F.col("product_cost").cast("string"),
            F.col("product_line")
        ), 256)
    )

    # Step 2: Hash business columns on existing Silver current records only
    df_existing = silver.toDF().withColumn(
        "existing_hash",
        F.sha2(F.concat_ws("||",
            F.col("product_name"),
            F.col("product_cost").cast("string"),
            F.col("product_line")
        ), 256)
    ).filter(F.col("is_current") == True) \
     .select("product_id", "existing_hash")

    # Step 3: Classify incoming records
    df_classified = df_incoming.join(df_existing, on="product_id", how="left")

    df_new     = df_classified.filter(F.col("existing_hash").isNull())
    df_changed = df_classified.filter(
                     F.col("existing_hash").isNotNull() &
                     (F.col("incoming_hash") != F.col("existing_hash"))
                 )

    # Step 4: Close changed records in Silver
    if not df_changed.isEmpty():
        changed_ids = [r["product_id"] for r in df_changed.select("product_id").collect()]

        silver.update(
            condition=(
                (F.col("is_current") == True) &
                (F.col("product_id").isin(changed_ids))
            ),
            set={
                "is_current": F.lit(False),
                "end_date":   today
            }
        )

    # Step 5: Insert new + changed records with correct version numbers
    df_to_insert = df_new.unionByName(df_changed)

    if not df_to_insert.isEmpty():
        df_max_version = silver.toDF() \
            .groupBy("product_id") \
            .agg(F.max("product_version").alias("max_version"))

        df_to_insert = df_to_insert \
            .join(df_max_version, on="product_id", how="left") \
            .withColumn(
                "product_version",
                F.coalesce(F.col("max_version"), F.lit(0)) + F.lit(1)
            ) \
            .drop("max_version", "incoming_hash", "existing_hash")

        df_to_insert.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TARGET_TABLE)

    print(f"SCD2 write complete — New: {df_new.count()}, Changed: {df_changed.count()}")